# torch.compile 与图捕获

## 学习目标

在支持的 PyTorch 版本上运行 `torch.compile`，比较 eager 与 compiled 模式，并识别动态 Python 控制流导致的 graph break。

## 概念模型

compile 会捕获并优化 Tensor 运算图。首次调用通常有编译开销，性能结论必须在预热后比较；不支持的 Python 行为可能触发 graph break 或回退。

In [ ]:
import torch
from torch import nn

model = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 2))
inputs = torch.randn(4, 8)
print('torch:', torch.__version__, 'compile available:', hasattr(torch, 'compile'))
eager = model(inputs)
assert eager.shape == (4, 2)

### 实验 1：编译模型并验证输出

小模型的首次编译可能比 eager 更慢，因此这里重点验证行为一致性。

In [ ]:
if hasattr(torch, 'compile'):
    compiled = torch.compile(model, backend='eager')
    compiled_output = compiled(inputs)
    torch.testing.assert_close(compiled_output, eager)
    print('compiled output:', compiled_output.shape)
else:
    print('torch.compile unavailable; skipped')

### 实验 2：动态控制流的边界

依赖 Tensor 值的 Python 分支可能造成 graph break；优先改写为 Tensor 运算，或接受必要的回退。

In [ ]:
class DynamicModel(nn.Module):
    def forward(self, x):
        if x.sum() > 0:
            return x * 2
        return x - 2

dynamic = DynamicModel()
if hasattr(torch, 'compile'):
    dynamic_compiled = torch.compile(dynamic, backend='eager')
    print('dynamic result:', dynamic_compiled(torch.ones(2)))
else:
    print('compile skipped')

## 检查点

解释首次编译开销、预热、graph break 和输出一致性验证。

## 试一试

用 `torch.profiler` 比较多次 eager/compiled 调用；尝试不同 batch size 并记录是否出现重新编译。

## 常见错误与调试

只测首次调用、把小模型的编译时间当作稳定收益、未验证数值一致性、忽略动态 shape 和 Python 控制流。